## Exploring architectural design choices

The structure should be this:
- I should have a file `models/` and each of this implements a single model. Starts with gpt and ends with recursion, slowly adding/removing one component up until a full recursion process is done.
- I should have a training task where I select which model is trained on and what dataset is trained on
- The training of the model and dataset should be independent arguments and I should be able to just select which one I am doing: which model on which dataset.
- The evaluation should also be clear, with the ability to evaluate both the losses on the val dataset as well as performance (e.g. if it's a math task) by just selecting the best model
- The saving of the files should also be clear, i.e. the best model should be saved.

What we need to fix:
- Same dataset
- Same prediction task
- Same loss
- Match parameter counts

So setup where:
- Different models
- Different datasets

---
What are the different models I have?
1. Model 1: Vanilla GPT
2. Model 2: Two-stream GPT
    - What is the effect of having a two-stream architecture, even without recursion? 
    - We add reasoning <- f_theta (reasoning, solution + input_seq)
    - We add solution <- f_theata (solution, reasoning)
    - GPT mixes everything into a single stream. M2 forces information to bounce between reasoning and solution.
3. Model 3: Reuse the same UpdateNEtwork multiple times per layer but keep gradients through all cycles to see pure recurrsence effects. Does “more depth via recurrence of the same block” help compared to a feedforward stack at fixed compute?
4. Model 4: Many recurrences cycles and one update
5. Model 5: Many currences cycles, one update, repeated multiple times
6. Model 6: Adding halting logic
---
Model 1: Vanilla GPT
- Key change: Standard single-stream Transformer decoder with causal self-attention and a feedforward stack; no recursion or auxiliary state.
- Purpose: Establish a strong, well-understood baseline for next-token prediction against which all recursive variants can be compared in terms of accuracy and compute.

Model 2: Two-Stream GPT (Solution–Reasoning Split)
- Key change: Replace the single hidden state with two coupled streams, updating reasoning ← fθ(reasoning, solution + input_seq) and solution ← fθ(solution, reasoning) once per layer, without any extra recurrence.
- Purpose: Isolate the effect of explicitly separating “working memory” (reasoning) from “answer representation” (solution) while holding total depth and compute similar to vanilla GPT.

Model 3: GPT with Recurrent Depth (Shared Update Network)
- Key change: Reuse the same UpdateNetwork multiple times per layer (inner cycles over reasoning and solution) with gradients flowing through all cycles, effectively increasing depth without adding new parameters.
- Purpose: Test whether greater effective depth via recurrence of a shared block improves modeling capacity compared to a purely feedforward stack at similar parameter count and compute.

Model 4: TR-GPT Warm-Up Recurrence (Many Cycles, One Gradient-Carrying Update)
- Key change: Introduce TRM-style recurrence where several outer cycles run under no_grad (warm-up) and only the final cycle carries gradients, with each cycle performing multiple reasoning updates followed by a solution update.
- Purpose: Probe whether cheap, gradient-free iterative refinement toward a fixed point before a single learned update helps or hurts language modeling performance.

Model 5:TR-GPT Multi-Step Supervision with Fixed Recurrence (Many Cycles, Repeated Updates)
- Key change: Keep the TRM warm-up recurrence per step, but apply it multiple times in an outer supervision loop, reusing the latent (solution, reasoning) state and optionally applying deep supervision across steps.
- Purpose: Study whether training the model to iteratively refine its solution over several supervised steps (with persistent latent state) yields better representations than a single-step prediction.

Model 6: TR-GPT with ACT-Style Halting (Adaptive Computation)
- Key change: Add a halting head on the solution stream and ACT-style logic that decides, per example, how many outer supervision steps to run up to a maximum, optionally regularized toward shorter computation.
- Purpose: Examine whether adaptive compute—allocating more recursion to difficult inputs and less to easy ones—improves the accuracy–compute trade-off compared to fixed-depth recursive models.


# Data visualization on algorithmic reasoning

In [2]:
import torch
import matplotlib.pyplot as plt

# from datasets.copy_char import load_copy_char
from data_modules.algorithmic_char import load_algorithmic_char


/home/azureuser/miniconda/envs/trm2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
BATCH_SIZE = 4
BLOCK_SIZE = 32
SEED = 42

train_loader, val_loader, tokenizer = load_algorithmic_char(
    task="reverse", # reverse, addition, copy
    data_dir="data",
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    eval_iters=100,
    seed=SEED,
    train_seq_len=3
)

In [10]:
# show some examples
X, Y = next(iter(train_loader))

# Show shapes
print(X.shape)
print(Y.shape)



torch.Size([4, 32])
torch.Size([4, 32])


In [11]:
print("---- X ----")
print(tokenizer.decode(X[0]))
print("---- Y----")
print(tokenizer.decode(Y[0]))

# Show tokenizer vocab size
#print(tokenizer.vocab_size)



---- X ----
|558
083|380
577|775
329|923
699
---- Y----
558
083|380
577|775
329|923
699|


In [8]:
# Compare X and Y for the first few characters of the first sample
sample_idx = 0
x_tokens = X[sample_idx].tolist()
y_tokens = Y[sample_idx].tolist()

print("\n--- Next-Token Prediction Verification ---")
for t in range(5): # Check first 5 positions
    curr_char = tokenizer.decode([x_tokens[t]])
    target_char = tokenizer.decode([y_tokens[t]])
    print(f"Input: '{curr_char}' -> Target: '{target_char}'")


--- Next-Token Prediction Verification ---
Input: '=' -> Target: '0'
Input: '0' -> Target: '5'
Input: '5' -> Target: '6'
Input: '6' -> Target: '0'
Input: '0' -> Target: '
'


In [ ]:
# Understanding the evaluation
